# ***Clustering Audio Using Dynamic Time Warping*** 

In [1]:
import os
import pandas as pd
import numpy as np

from pydub import AudioSegment

import librosa
import tsfel


## **Convert Data Audio**

In [2]:
def convert_to_wav(input_path, output_path):
    """Konversi file audio apapun (AAC, M4A, MP3) menjadi WAV."""
    audio = AudioSegment.from_file(input_path)
    audio = audio.set_frame_rate(22050)  # Sample rate standar librosa
    audio.export(output_path, format="wav")
    return output_path

In [3]:
RAW_DIR = "dataset/audio"
PROCESSED_DIR = "dataset/processed_audio_fauzan"

# os.makedirs(PROCESSED_DIR, exist_ok=True)

In [4]:
def batch_convert_audio():
    for label in os.listdir(RAW_DIR):

        raw_folder = os.path.join(RAW_DIR, label)

        # skip jika raw_folder adalah file, bukan folder
        if not os.path.isdir(raw_folder):
            continue

        processed_folder = os.path.join(PROCESSED_DIR, label.replace(" ", "_"))
        os.makedirs(processed_folder, exist_ok=True)

        for fname in os.listdir(raw_folder):
            input_path = os.path.join(raw_folder, fname)

            # skip semua item yang bukan file (folder akan dilewati)
            if not os.path.isfile(input_path):
                continue

            # hanya proses file audio yang valid
            ext = os.path.splitext(fname)[1].lower()
            if ext not in [".aac", ".m4a", ".mp3", ".wav"]:
                print(f"Skipping non-audio: {fname}")
                continue

            output_name = os.path.splitext(fname)[0] + ".wav"
            output_path = os.path.join(processed_folder, output_name)

            if ext == ".wav":
                continue  # jika sudah wav, tidak perlu konversi

            print(f"Mengkonversi: {input_path}")
            convert_to_wav(input_path, output_path)

    print("Konversi selesai!")

In [5]:
# batch_convert_audio()

## **Membaca Dataset Audio**

In [6]:
PROCESSED_DIR = "dataset/audio/processed_audio_fauzan"

In [7]:
def load_all_audio():
    audio_data = {}

    for label in os.listdir(PROCESSED_DIR):
        folder_path = os.path.join(PROCESSED_DIR, label)

        # skip jika bukan folder
        if not os.path.isdir(folder_path):
            continue

        print(f"Loading folder: {label}")
        audio_data[label] = []

        for fname in os.listdir(folder_path):
            if not fname.lower().endswith(".wav"):
                continue  # hanya terima wav

            file_path = os.path.join(folder_path, fname)

            # Load audio dengan librosa
            signal, sr = librosa.load(file_path, sr=22050)

            audio_data[label].append({
                "filename": fname,
                "signal": signal,
                "sr": sr
            })

            print(f"  Loaded: {fname}, length={len(signal)} samples")

    return audio_data


In [8]:
audio_dataset = load_all_audio()

Loading folder: buka_fauzan
  Loaded: 20251030_183541.wav, length=37888 samples
  Loaded: 20251030_183623.wav, length=29184 samples
  Loaded: 20251030_183635.wav, length=61440 samples
  Loaded: 20251030_183717.wav, length=39936 samples
  Loaded: 20251030_183723.wav, length=36352 samples
  Loaded: 20251030_183736.wav, length=43008 samples
  Loaded: 20251030_183740.wav, length=47616 samples
  Loaded: 20251030_183748.wav, length=29696 samples
  Loaded: 20251030_183754.wav, length=34816 samples
  Loaded: 20251030_183813.wav, length=30720 samples
  Loaded: 20251030_183817.wav, length=40960 samples
  Loaded: 20251030_183835.wav, length=34304 samples
  Loaded: 20251030_183938.wav, length=31232 samples
  Loaded: 20251030_183942.wav, length=41472 samples
  Loaded: 20251030_183947.wav, length=38912 samples
  Loaded: 20251030_183957.wav, length=32768 samples
  Loaded: 20251030_184003.wav, length=37888 samples
  Loaded: 20251030_184011.wav, length=32256 samples
  Loaded: 20251030_184015.wav, lengt

## **Ekstraksi Fitur dengan Librosa dan TSFEL**

In [9]:
def extract_features_librosa(signal, sr):
    """Ekstrak fitur audio menggunakan Librosa.
       Output = 1 vektor fitur panjang tetap.
    """

    features = []

    # --- MFCC ---
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std  = np.std(mfcc, axis=1)
    features.extend(mfcc_mean)
    features.extend(mfcc_std)

    # --- Spectral centroid ---
    centroid = librosa.feature.spectral_centroid(y=signal, sr=sr)
    features.append(np.mean(centroid))

    # --- Spectral bandwidth ---
    bandwidth = librosa.feature.spectral_bandwidth(y=signal, sr=sr)
    features.append(np.mean(bandwidth))

    # --- Zero-crossing rate ---
    zcr = librosa.feature.zero_crossing_rate(signal)
    features.append(np.mean(zcr))

    # --- RMS energy ---
    rms = librosa.feature.rms(y=signal)
    features.append(np.mean(rms))

    return np.array(features)


In [10]:
def extract_features_tsfel(signal, sr):
    """Ekstrak fitur menggunakan TSFEL.
       Mengembalikan 1 vektor fitur (numpy).
    """

    # convert ke dataframe (tsfel butuh DF/series)
    df_signal = pd.DataFrame(signal)

    # ambil konfigurasi default (time, statistical, spectral)
    cfg = tsfel.get_features_by_domain()

    # ekstraksi fitur
    extracted = tsfel.time_series_features_extractor(
        cfg, df_signal, fs=sr, verbose=0
    )

    # ubah dataframe → numpy vector
    vec = extracted.iloc[0].values

    return vec

In [11]:
def build_feature_dataset(audio_dataset, method="librosa"):
    X = []
    y = []

    for label, items in audio_dataset.items():
        for item in items:
            sig = item["signal"]
            sr = item["sr"]

            if method == "librosa":
                feat = extract_features_librosa(sig, sr)
            elif method == "tsfel":
                feat = extract_features_tsfel(sig, sr)

            X.append(feat)
            y.append(label)

    return np.array(X), np.array(y)


In [12]:
X_librosa, y_librosa = build_feature_dataset(audio_dataset, method="librosa")

In [13]:
X_tsfel, y_tsfel = build_feature_dataset(audio_dataset, method='tsfel')

## Gak tau